# Motor de Busca Semântica para Normas de Graduação da UFMG

Este notebook implementa um sistema de recuperação de informação em três estágios para responder dúvidas de alunos sobre normas da UFMG:

1. **Busca híbrida** (E5-large + BM25): recupera os 10 artigos mais prováveis
2. **Reranking** (Cross-Encoder): reordena os candidatos lendo pergunta e artigo juntos
3. **Avaliação** (Recall@k e MRR): mede a qualidade do sistema com 316 perguntas anotadas

---

***AVISOS***:
1) Para executar o código, são necessários dois arquivos: regimento_indexado.txt e perguntas&respostas2.txt. Ambos foram enviados junto ao código e relatório. É necessário fazer upload deles na célula 2.
2) A interface visual usando Gradio está na última célula do notebook. Ela pode também ser acessada pelo link https://e8f95aca8c84ea0251.gradio.live, porém há um limite de 7 dias para o funcionamento do link, logo pode não funcionar caso a correção seja muito após a data em que entregamos.

## 1. Instalação de Dependências

In [28]:
!pip install sentence-transformers faiss-cpu pandas rank-bm25

## 2. Carregamento dos Arquivos

São necessários dois arquivos:
- `regimento_indexado.txt`: artigos do Regimento Geral da UFMG, cada um precedido por `#index N`
- `perguntas&respostas2.txt`: perguntas em linguagem natural, cada uma anotada com o `#index` do artigo correto

In [29]:
from google.colab import files

arquivos = files.upload()

Saving perguntas&respostas2.txt to perguntas&respostas2 (1).txt
Saving regimento_indexado.txt to regimento_indexado (1).txt


In [30]:
arquivos.keys()

dict_keys(['perguntas&respostas2 (1).txt', 'regimento_indexado (1).txt'])

## 3. Parsing do Regimento

O arquivo de normas usa o marcador `#index N` para delimitar cada artigo. A expressão regular abaixo extrai todos os artigos e os armazena num dicionário `normas`, onde a chave é o índice numérico e o valor é o texto do artigo.

In [31]:
with open("regimento_indexado.txt", "r", encoding="utf-8") as f:
    texto = f.read()

print(texto[:1000])

﻿#index 0
Art. 1o Aprovar as seguintes alterações no Regimento Geral da Universidade Federal de Minas Gerais, consolidadas no ANEXO:
 
                            I - Artigo 95, caput
                            Onde se lê:
                            
#index 1
Art. 95. A UFMG reconhecerá como órgão de representação do corpo discente, no plano da Universidade, o Diretório Central dos Estudantes-DCE, e, no plano das Unidades, os Diretórios Acadêmicos-DAs ou Centros Acadêmicos-CAs, entidades autônomas organizadas nos termos dos respectivos estatutos, aprovados na forma da lei.
                                Leia-se:
                            
#index 2
Art. 95. A UFMG reconhecerá como órgão de representação do corpo discente, no plano da Universidade, o Diretório Central dos Estudantes-DCE, e, no plano das Unidades, os Diretórios Acadêmicos-DAs ou Centros Acadêmicos-CAs, entidades autônomas organizadas nos termos dos respectivos estatutos e cujas atas de eleição e posse de seus dirigen

In [32]:
import re

with open("regimento_indexado.txt", "r", encoding="utf-8") as f:
    regimento = f.read()


padrao = r'#index (\d+)\n(.*?)(?=\n#index \d+|$)'


artigos = re.findall(
    padrao,
    regimento,
    flags=re.DOTALL
)


normas = {}

for indice, texto in artigos:
    normas[int(indice)] = texto.strip()


normas[121]

'Art. 117. O pedido de revisão, seja por solicitação de reconsideração, seja por interposição de recurso, tramitará, no máximo, por três instâncias administrativas, salvo disposição normativa diversa.'

## 4. Parsing do Dataset de Avaliação

O arquivo de perguntas contém 316 perguntas geradas em linguagem natural informal, cada uma anotada com o índice do artigo que contém a resposta correta. Esse par *(pergunta, índice_esperado)* será usado para calcular as métricas ao final.

In [33]:
with open("perguntas&respostas2.txt", "r", encoding="utf-8-sig") as f:
    perguntas_txt = f.read()


padrao = r'(.+?)\s*#index\s+(\d+)'


perguntas = re.findall(
    padrao,
    perguntas_txt,
    flags=re.DOTALL
)

perguntas = [
    (pergunta.replace("\\n", "").strip(), int(indice))
    for pergunta, indice in perguntas
]

In [34]:
perguntas[0][0]

'Onde que eu acho o texto com as últimas mudanças que a UFMG aprovou para o Regimento Geral?'

## 5. Modelo de Embeddings: multilingual-E5-large

Utilizamos o modelo `intfloat/multilingual-e5-large` para gerar os embeddings. Ele foi escolhido por ter sido treinado especificamente para busca assimétrica, onde a query é estruturalmente diferente do documento. Nesse projeto temos perguntas informais e respostas (normas) em texto jurídico formal, logo são estruturalmente diferentes.



In [35]:
from sentence_transformers import SentenceTransformer

In [36]:
modelo = SentenceTransformer(
    "intfloat/multilingual-e5-large"
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## 6. Indexação Semântica com FAISS

Cada artigo é convertido em um vetor de 1024 dimensões pelo E5-large. Os vetores são normalizados (norma L2 = 1) e indexados no FAISS usando produto interno (IndexFlatIP), que equivale à similaridade por cosseno após a normalização.

In [37]:
indices = list(normas.keys())

textos_normas = [
    "passage: " + normas[i] for i in indices
]

In [38]:
indices[:5]

[0, 1, 2, 3, 4]

In [39]:
embeddings_normas = modelo.encode(
    textos_normas,
    convert_to_numpy=True
)

In [40]:
embeddings_normas.shape

(159, 1024)

In [41]:
import faiss

In [42]:
import numpy as np

embeddings_normas = embeddings_normas.astype("float32")

faiss.normalize_L2(embeddings_normas)

indice_faiss = faiss.IndexFlatIP(
    embeddings_normas.shape[1]
)

indice_faiss.add(embeddings_normas)

## 7. Indexação Lexical com BM25

BM25 é um modelo clássico de recuperação de informação baseado em frequência de termos. Ele complementa o E5 em situações onde a pergunta contém termos jurídicos específicos (como *'trancamento'*, *'jubilamento'*, *'colação de grau'*) que aparecem literalmente no artigo. O score final de cada artigo será a média ponderada entre o score semântico (E5) e o score lexical (BM25).

In [43]:
from rank_bm25 import BM25Okapi

# BM25 sobre os textos originais (sem prefixo) para capturar termos exatos
corpus_tokenizado = [normas[i].lower().split() for i in indices]
bm25 = BM25Okapi(corpus_tokenizado)

print("BM25 indexado com", len(corpus_tokenizado), "normas")

BM25 indexado com 159 normas


## 8. Exemplo de Busca Híbrida

Antes da avaliação formal, testamos o pipeline com uma pergunta de exemplo para verificar o funcionamento. Os scores de cada componente são exibidos para fins de diagnóstico.

In [44]:
pergunta_teste = "O que causa desligamento da universidade?"

In [45]:
# O modelo E5 exige o prefixo 'query:' na pergunta
vetor_pergunta = modelo.encode(
    ["query: " + pergunta_teste],
    convert_to_numpy=True
)

vetor_pergunta = vetor_pergunta.astype("float32")

faiss.normalize_L2(vetor_pergunta)

In [46]:
distancias, resultados = indice_faiss.search(
    vetor_pergunta,
    k=10  # busca mais candidatos para o reranking híbrido
)

In [47]:
import numpy as np

# --- Scores semânticos (FAISS / E5) ---
scores_semanticos = np.zeros(len(indices))
for rank, posicao in enumerate(resultados[0]):
    scores_semanticos[posicao] = distancias[0][rank]

# --- Scores lexicais (BM25) ---
tokens_pergunta = pergunta_teste.lower().split()
scores_bm25_raw = bm25.get_scores(tokens_pergunta)

# Normaliza BM25 para [0, 1] antes de combinar
bm25_max = scores_bm25_raw.max()
scores_bm25_norm = scores_bm25_raw / bm25_max if bm25_max > 0 else scores_bm25_raw

# --- Score híbrido (50% semântico + 50% lexical) ---
alpha = 0.5
scores_hibridos = alpha * scores_semanticos + (1 - alpha) * scores_bm25_norm

# --- Top-5 resultados finais ---
top5 = np.argsort(scores_hibridos)[::-1][:5]

for posicao in top5:
    idx_norma = indices[posicao]
    print(f"INDEX: {idx_norma} | score_hibrido={scores_hibridos[posicao]:.3f} "
          f"(semantico={scores_semanticos[posicao]:.3f}, bm25={scores_bm25_norm[posicao]:.3f})")
    print(normas[idx_norma])
    print("----------------")

INDEX: 45 | score_hibrido=0.923 (semantico=0.847, bm25=1.000)
Art. 41. A permanência do aluno na UFMG dar-se-á até:
I - a conclusão do curso e a obtenção do grau acadêmico;
II - o desligamento e o consequente cancelamento do registro acadêmico, por:
a) descumprimento de exigências previstas nas Normas Gerais de Graduação e nas Normas Gerais de Pós-Graduação;
b) aplicação pela Universidade das condições de desligamento previstas nas Normas Gerais de Graduação e nas Normas Gerais de Pós-Graduação considerada a condição pública da vaga ocupada;
c) aplicação de penalidade prevista no Código de Convivência Discente.
III - a desistência formal da vaga a que tem direito.
CAPÍTULO II
Da Graduação
----------------
INDEX: 75 | score_hibrido=0.892 (semantico=0.819, bm25=0.965)
Art. 71. Os títulos de Doutor Honoris Causa e de Professor Honoris Causa não são concedidos a servidor da UFMG, seja do corpo docente, seja do corpo técnico-administrativo em educação, mesmo aposentado.
----------------
IND

## 9. Avaliação do Pipeline Híbrido (sem reranking)

Avaliamos o sistema sobre todas as 316 perguntas anotadas usando três métricas:

- **Recall@k**: proporção de perguntas em que o artigo correto aparece entre os top-k resultados
- **MRR (Mean Reciprocal Rank)**: média do inverso da posição do artigo correto — penaliza mais quando o acerto aparece em posições mais baixas. MRR = 1.0 seria perfeito (sempre na posição 1); MRR = 0.5 significa que em média o acerto está na posição 2.

Nesta etapa, avaliamos apenas a busca híbrida **sem** o reranker, para isolar a contribuição de cada estágio.

In [48]:
def buscar_hibrido(pergunta, k=10):
    vetor = modelo.encode(
        ["query: " + pergunta],
        convert_to_numpy=True
    ).astype("float32")
    faiss.normalize_L2(vetor)

    distancias, resultados = indice_faiss.search(vetor, k=k)

    scores_semanticos = np.zeros(len(indices))
    for rank, posicao in enumerate(resultados[0]):
        scores_semanticos[posicao] = distancias[0][rank]

    tokens = pergunta.lower().split()
    scores_bm25_raw = bm25.get_scores(tokens)
    bm25_max = scores_bm25_raw.max()
    scores_bm25_norm = scores_bm25_raw / bm25_max if bm25_max > 0 else scores_bm25_raw

    scores_hibridos = 0.5 * scores_semanticos + 0.5 * scores_bm25_norm
    top_k = np.argsort(scores_hibridos)[::-1][:k]
    return [indices[pos] for pos in top_k]


acertos_top1 = 0
acertos_top5 = 0
acertos_top10 = 0
soma_reciprocal_rank = 0.0
erros = []

for pergunta, index_correto in perguntas:
    resultados = buscar_hibrido(pergunta, k=10)

    if index_correto == resultados[0]:
        acertos_top1 += 1
    if index_correto in resultados[:5]:
        acertos_top5 += 1
    if index_correto in resultados[:10]:
        acertos_top10 += 1
        # MRR: reciprocal rank = 1 / posição (base 1)
        posicao = resultados.index(index_correto) + 1
        soma_reciprocal_rank += 1.0 / posicao
    else:
        erros.append((pergunta, index_correto, resultados[:5]))

total = len(perguntas)
mrr = soma_reciprocal_rank / total

print(f"Total de perguntas: {total}")
print(f"Recall@1:  {acertos_top1}/{total} = {acertos_top1/total:.1%}")
print(f"Recall@5:  {acertos_top5}/{total} = {acertos_top5/total:.1%}")
print(f"Recall@10: {acertos_top10}/{total} = {acertos_top10/total:.1%}")
print(f"MRR@10:    {mrr:.4f}")

print("\n--- Exemplos de erro (sem reranking) ---")
for pergunta, correto, obtidos in erros[:5]:
    print(f"Pergunta: {pergunta}")
    print(f"Esperado: #{correto} → {normas[correto][:100]}...")
    print(f"Obtidos:  {obtidos}")
    print()

Total de perguntas: 316
Recall@1:  147/316 = 46.5%
Recall@5:  248/316 = 78.5%
Recall@10: 291/316 = 92.1%
MRR@10:    0.6117

--- Exemplos de erro (sem reranking) ---
Pergunta: O que acontece com as resoluções antigas, tipo aquela de 2018 sobre a representação estudantil, quando sai uma norma nova?
Esperado: #3 → Art. 2o Revogam-se as disposições em contrário, em especial a Resolução Complementar no 03/2018, de ...
Obtidos:  [105, 22, 80, 7, 103]

Pergunta: Como eu sei se uma regra antiga da UFMG sobre os estudantes perdeu a validade por causa de outra mais nova?
Esperado: #3 → Art. 2o Revogam-se as disposições em contrário, em especial a Resolução Complementar no 03/2018, de ...
Obtidos:  [118, 2, 43, 102, 136]

Pergunta: Quem são as autoridades ou conselhos que têm o poder de criar as resoluções que mandam no funcionamento da UFMG?
Esperado: #6 → Art. 2o O Conselho Universitário, o Conselho de Ensino, Pesquisa e Extensão-CEPE e o Colegiado Super...
Obtidos:  [118, 95, 155, 138, 14]

Pe

## 10. Reranking com Cross-Encoder

O reranker `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` lê a pergunta e cada artigo candidato **juntos no mesmo contexto**, produzindo um score de relevância muito mais preciso do que comparar vetores separados. O pipeline final é:

1. Busca híbrida (E5 + BM25) → top-10 candidatos
2. Cross-encoder reordena os 10 candidatos
3. O artigo no topo é retornado como resposta

O custo computacional do cross-encoder é maior, mas como ele opera apenas sobre 10 candidatos por pergunta (e não sobre as 159 normas inteiras), o tempo total permanece viável.

In [49]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [50]:
def buscar_com_reranking(pergunta, k=10):
    candidatos_indices = buscar_hibrido(pergunta, k=k)
    candidatos_textos = [normas[i] for i in candidatos_indices]

    pares = [(pergunta, texto) for texto in candidatos_textos]
    scores = reranker.predict(pares)

    ordem = np.argsort(scores)[::-1]
    return [candidatos_indices[i] for i in ordem]


acertos_top1 = 0
acertos_top5 = 0
soma_reciprocal_rank = 0.0
erros_reranker = []

for pergunta, index_correto in perguntas:
    resultados = buscar_com_reranking(pergunta, k=10)

    if index_correto == resultados[0]:
        acertos_top1 += 1
    if index_correto in resultados[:5]:
        acertos_top5 += 1

    # MRR: se o correto não estiver nos top-10, reciprocal rank = 0
    if index_correto in resultados:
        posicao = resultados.index(index_correto) + 1
        soma_reciprocal_rank += 1.0 / posicao
    else:
        erros_reranker.append((pergunta, index_correto, resultados[:5]))

total = len(perguntas)
mrr = soma_reciprocal_rank / total

print(f"Total de perguntas: {total}")
print(f"Recall@1  com reranking: {acertos_top1}/{total} = {acertos_top1/total:.1%}")
print(f"Recall@5  com reranking: {acertos_top5}/{total} = {acertos_top5/total:.1%}")
print(f"MRR@10    com reranking: {mrr:.4f}")

print("\n--- Exemplos de erro (com reranking) ---")
for pergunta, correto, obtidos in erros_reranker[:5]:
    print(f"Pergunta: {pergunta}")
    print(f"Esperado: #{correto} → {normas[correto][:100]}...")
    print(f"Obtidos:  {obtidos}")
    print()

Total de perguntas: 316
Recall@1  com reranking: 233/316 = 73.7%
Recall@5  com reranking: 286/316 = 90.5%
MRR@10    com reranking: 0.8111

--- Exemplos de erro (com reranking) ---
Pergunta: O que acontece com as resoluções antigas, tipo aquela de 2018 sobre a representação estudantil, quando sai uma norma nova?
Esperado: #3 → Art. 2o Revogam-se as disposições em contrário, em especial a Resolução Complementar no 03/2018, de ...
Obtidos:  [157, 2, 105, 80, 156]

Pergunta: Como eu sei se uma regra antiga da UFMG sobre os estudantes perdeu a validade por causa de outra mais nova?
Esperado: #3 → Art. 2o Revogam-se as disposições em contrário, em especial a Resolução Complementar no 03/2018, de ...
Obtidos:  [45, 2, 102, 155, 118]

Pergunta: Quem são as autoridades ou conselhos que têm o poder de criar as resoluções que mandam no funcionamento da UFMG?
Esperado: #6 → Art. 2o O Conselho Universitário, o Conselho de Ensino, Pesquisa e Extensão-CEPE e o Colegiado Super...
Obtidos:  [138, 95, 1

## 11. Comparação dos Estágios do Pipeline

A tabela abaixo consolida o impacto de cada estágio do pipeline nos resultados finais.

In [51]:
import pandas as pd

tabela = pd.DataFrame({
    "Estágio": ["E5 + BM25 (sem reranker)", "E5 + BM25 + Cross-Encoder"],
    "Recall@1": ["46.5%", "73.7%"],
    "Recall@5": ["78.5%", "90.5%"],
    "Recall@10": ["92.1%", "92.1%"],
})

print(tabela.to_string(index=False))

                  Estágio Recall@1 Recall@5 Recall@10
 E5 + BM25 (sem reranker)    46.5%    78.5%     92.1%
E5 + BM25 + Cross-Encoder    73.7%    90.5%     92.1%


## 12. Célula para teste de perguntas no modelo final

In [53]:
perguntas_demo = [
    "Faltei muito nas aulas, posso ser reprovado por isso?",
    "Tirei nota 60, passei ou fui reprovado?",
    "Quero fazer outro curso aqui na UFMG, como faço?"
]

for pergunta in perguntas_demo:
    resultados = buscar_com_reranking(pergunta, k=10)
    artigo_top1 = normas[resultados[0]]

    print(f"Pergunta: {pergunta}")
    print(f"Artigo #{resultados[0]}:")
    print(artigo_top1)
    print("=" * 60)

Pergunta: Faltei muito nas aulas, posso ser reprovado por isso?
Artigo #52:
Art. 48. O desempenho acadêmico escolar do aluno será verificado em cada atividade acadêmica curricular, abrangendo os aspectos de assiduidade e aproveitamento, cada um dos quais com caráter reprovatório.
§ 1o A assiduidade mínima obrigatória, em cada atividade acadêmica curricular, é de 75% da carga horária prevista, exceto nos casos estabelecidos em lei.
§ 2o A verificação do desempenho do aluno será feita por pontos cumulativos, em uma escala de zero a cem.
Pergunta: Tirei nota 60, passei ou fui reprovado?
Artigo #53:
Art. 49. Apurados os resultados finais, o desempenho acadêmico de cada aluno será convertido nos seguintes conceitos:
I - A: de 90 a 100 pontos;
II - B: de 80 a 89 pontos;
III - C: de 70 a 79 pontos;
IV - D: de 60 a 69 pontos;
V - E: de 40 a 59 pontos;
VI - F: abaixo de 40 pontos de aproveitamento e/ou inassiduidade do aluno.
Parágrafo único. O aluno assíduo que alcançar, no mínimo, conceito D 

## 13. Interface Gráfica (Gradio)

In [52]:
import gradio as gr

def buscar(pergunta):
    resultados = buscar_com_reranking(pergunta, k=10)
    artigo_top1 = normas[resultados[0]]
    return artigo_top1

interface = gr.Interface(
    fn=buscar,
    inputs=gr.Textbox(label="Sua pergunta"),
    outputs=gr.Textbox(label="Artigo encontrado", lines=15),
    title="Busca no Regimento da UFMG"
)

interface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e8f95aca8c84ea0251.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
